UniGuide_AI
Joud Saeed Alghamdi

In [22]:
!pip install -q \
langchain \
langgraph \
langchain-google-genai \
langchain-chroma \
chromadb \
langsmith

In [23]:
import os

os.environ["GOOGLE_API_KEY"] = ""

In [24]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash"
)

response = llm.invoke(
    "Reply with exactly: UniGuide AI is ready."
)

print(response.content)

[{'type': 'text', 'text': 'UniGuide AI is ready.', 'extras': {'signature': 'EuYDCuMDARFNMg9On1ITuroses7ytDT8MHgGI8bGIN/dklp4+YER4H6X4S7+PAlkNnA769NfGl9zCdSlWaUL8LWotH+5emRqSY+XLzdxVpt0hH0vQCph6CzBQCvu+z2N1TA4uqy6oZcd7R9sH5dsMB+wB9ia0Rp4RDKGufBAsHuRivvKe9MLMByJm0EFMfGJ6umSrgyYUMw7tJviyOBwYzDPCM6h8QDzO7ywkqzxYgQSWi5J04nBUM4BSokBmH6lsY4QppyCyBt3q3nPvopyuIihWlyC010rt184kutcxy0YoLzBjA0SnokBzZnc5k5ie8dtM8uFiOqCcwE/+/cZggFbcsu+xiqD+g0NFkOXHD5XjsLdNAXNQjEb4KVws9zbK34679H1Qfwzu/rjmscWoh1Ej85Z79LNMJI8qK+PLnNmuqt/4A5VPcGOORKPN6rYCZpULpFD8vxCqWI2xjIOBhMkF2cWzmYfSYna6xO7FEA43a2ZuET0FXvgIs4Rzq+WaChlbHpxuODFSK2kmNtWx0qhaaIbLRpjhjOW4XKJ1hMvnuSsTwWDe7DqSJGAoOPQ1kW7g+Vy+pxvXmcdy2vfKke8rq8wrsc3bCFPMSZkU511UpjIU1Jsr/ZK+P+302Ixp+roo6auecQg'}}]


In [25]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="The Academic Regulations explain university rules, grading, attendance, and academic policies.",
        metadata={"source": "Academic Regulations"}
    ),
    Document(
        page_content="Course Information contains details about modules, prerequisites, and assessments.",
        metadata={"source": "Course Information"}
    ),
    Document(
        page_content="Campus Services include the library, IT support, careers service, and student wellbeing.",
        metadata={"source": "Campus Services"}
    ),
]

print(f"Loaded {len(documents)} documents.")

for doc in documents:
    print(f"- {doc.metadata['source']}")

Loaded 3 documents.
- Academic Regulations
- Course Information
- Campus Services


In [26]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings
)

print("Vector database created successfully!")

Vector database created successfully!


In [27]:
retriever = vectorstore.as_retriever()

print("Retriever is ready!")

Retriever is ready!


In [28]:
from langchain.tools import tool

@tool
def search_university_knowledge(question: str) -> str:
    """
    Search the university knowledge base for information about
    academic regulations, courses, assessments, and campus services.
    """
    docs = retriever.invoke(question)

    if not docs:
        return "No relevant university information was found."

    results = []

    for doc in docs:
        source = doc.metadata.get("source", "Unknown source")
        results.append(
            f"Source: {source}\n"
            f"Information: {doc.page_content}"
        )

    return "\n\n".join(results)

print("University search tool is ready!")

University search tool is ready!


In [29]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[search_university_knowledge],
    system_prompt="""
You are UniGuide AI, a university knowledge assistant.

Always use the search_university_knowledge tool before answering
questions about academic regulations, courses, assessments,
campus services, the library, IT support, or student wellbeing.

Answer only from the information returned by the tool.
Mention the source in your answer.
"""
)

print("Agent is ready!")

Agent is ready!


In [30]:
agent_result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What campus services are available?"
            }
        ]
    }
)

print(agent_result["messages"][-1].content)

[{'type': 'text', 'text': 'According to the **Campus Services** source, the available campus services include:\n* The library\n* IT support\n* Careers service\n* Student wellbeing', 'extras': {'signature': 'EtUCCtICARFNMg9y0LRJwU+ftXBEccJuwTw85trjNjPDOtg4dCrMOsHfQElp0xZlcbuGLQHDxOw7IYLkdrpDeWEmDwl5yBWYAuc3IVuEtVvoOwLha96vuRW9Wos3QJTxVSNHnl+cyQw2jGJJmn/0p+TBWRhrfOfhzp06foIMod62x0paE+w86u7ePI+lMhJWM9wSr/h92aI1WbX3X6WYMVE8U4MxfmZaBVwe3ym6gvaVIFtV9N+Rfqfxp3Z1uyKwlts9/je/iUxATKwM2mYpur1U210pr9WYPxHyWIzkSAb6qQTVvQmOnZI9HTKrsobmrV9wZwwOMxW3oYtWyEy4qAz4u/ONPV8mvXUJsxW51KZp+xFRWbSIooHJ9CdVgPqTHvSNgZzzMHq/9stoJGs6LqeSVaRwASfBYBLo9sR1DlHBLDtiA+049v1IQ8SJMKt6cdx7WiRmGME='}}]


In [31]:
query = "What campus services are available?"

results = retriever.invoke(query)

for doc in results:
    print(doc.page_content)
    print("Source:", doc.metadata["source"])
    print("-" * 50)

Campus Services include the library, IT support, careers service, and student wellbeing.
Source: Campus Services
--------------------------------------------------
Campus Services include the library, IT support, careers service, and student wellbeing.
Source: Campus Services
--------------------------------------------------
The Academic Regulations explain university rules, grading, attendance, and academic policies.
Source: Academic Regulations
--------------------------------------------------
The Academic Regulations explain university rules, grading, attendance, and academic policies.
Source: Academic Regulations
--------------------------------------------------


In [32]:
query = "What campus services are available?"

docs = retriever.invoke(query)

context = "\n\n".join([doc.page_content for doc in docs])

prompt = f"""
You are UniGuide AI.

Answer the user's question using ONLY the information below.

Context:
{context}

Question:
{query}
"""

response = llm.invoke(prompt)

print(response.content)

[{'type': 'text', 'text': 'Based on the provided information, the campus services available are:\n* The library\n* IT support\n* Careers service\n* Student wellbeing', 'extras': {'signature': 'EsUJCsIJARFNMg+9Ryq39l48GjyB120/JP1Db3UpK0urPU9VYx79iquDqKO0pC/yaml28Pa/SeX4lLabnEyQlrSfNNiHTSL8im/uLeMGgI+1pZK1U69oTpr11tfjZhnfgtAVrIYRFCS1uwg6vtUknkcI8tnfX1Oe8JYbYN4RXK+CFAFaGiOZpP3OxXeQnv+467K6Q46QOeBmVbgU33fhJtKCVm8OCHegBgtKbqAQKkFsWu2yyzCNqh9MnMAokP5s/DjcpJ8Xt9y2gW4QF3NCXc5S0sVRG6JhZIsBhdzviVYJvjEi4aDO0repep0iMYZKkqVSa5TLv56vOnBYLZ6YuZba/zMnL4vK8KoMP1adQI0YoZW6iIfcOTeflimEmHmKNUl4nRZJYEoX34yBjTs1suFbtSQuTJdQlgDtfpp+EUa2naakyIQ9eC+yJ7esIOD/vNwawegCqMkHPtCbXLd4+MIBmKGOqIAVvt0XwfkK04yCTlIeJ3JNGCPPpgWEWt2KFluVQpvXMziqlaqNnx4fb7dKadkqKeCHHPK4haEl5KseLX5yFtpAKLqh1fAZ/HQJEl65F2GjR8tpXCGzUE+9KNwWTO84gR2U+hQY1Ne+EATL25Co4kt+e1n+BUYA3vBbbQwKRD6MCP6wbNKpGSEVxslMtb7l8SAIXmLYBIZfE7opyIYGyzRyV8xQ4zuTMM7QY4iAtWf1KUBt3wgmhwdreuP6w1WotdXy51BoA/d7jiolnohVDhdWbdSDccmSaJm+cF+ax8Knn6W3VW853JOVqG0wFdhPHiNm640GjVB

In [33]:
import time

def rag_node(state: UniGuideState):
    question = state["question"]

    docs = retriever.invoke(question)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    prompt = f"""
You are UniGuide AI.

Answer the user's question using ONLY the information below.

Context:
{context}

Question:
{question}
"""

    max_attempts = 5

    for attempt in range(max_attempts):
        try:
            print(f"Gemini attempt: {attempt + 1}")

            response = llm.invoke(prompt)

            if isinstance(response.content, list):
                answer = response.content[0]["text"]
            else:
                answer = response.content

            return {"answer": answer}

        except Exception as error:
            if attempt < max_attempts - 1:
                wait_time = (attempt + 1) * 10

                print(
                    f"Gemini is busy. Retrying in {wait_time} seconds..."
                )

                time.sleep(wait_time)

            else:
                return {
                    "answer": (
                        "Gemini is temporarily unavailable because of high "
                        "demand. Please try again later."
                    )
                }

In [34]:
from typing_extensions import TypedDict
from typing import Literal

class UniGuideState(TypedDict):
    question: str
    category: str
    answer: str


def router(state: UniGuideState):
    question = state["question"].lower()

    if any(word in question for word in ["grade", "attendance", "regulation", "rules"]):
        category = "academic_regulations"

    elif any(word in question for word in ["course", "module", "assessment", "prerequisite"]):
        category = "course_information"

    else:
        category = "campus_services"

    print("Selected category:", category)

    return {"category": category}

In [35]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

graph = StateGraph(UniGuideState)

# Nodes
graph.add_node("router", router)
graph.add_node("rag", rag_node)

# Flow
graph.add_edge(START, "router")
graph.add_edge("router", "rag")
graph.add_edge("rag", END)

# Memory
memory = MemorySaver()

# Compile
app = graph.compile(checkpointer=memory)

print("LangGraph is ready!")

LangGraph is ready!


In [36]:
config = {
    "configurable": {
        "thread_id": "user-1"
    }
}

result = app.invoke(
    {
        "question": "What is the attendance policy?"
    },
    config=config
)

print("Category:", result["category"])
print("Answer:", result["answer"])

Selected category: academic_regulations
Gemini attempt: 1
Category: academic_regulations
Answer: Based on the provided context, the attendance policy is explained in the Academic Regulations.
